In [ ]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

df = pd.read_csv("/content/drive/MyDrive/Clean Data(LLM Responses ).csv")




Mounted at /content/drive


# Correctness - LLM

In [ ]:

import os
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "50000"
os.environ["DEEPEVAL_TASK_GATHER_BUFFER_SECONDS_OVERRIDE"] = "200"
import asyncio
from ollama import AsyncClient, Client
from deepeval.models import DeepEvalBaseLLM
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.metrics import GEval
# OS New Edits
from deepeval.metrics import (
  ContextualRelevancyMetric,
  ContextualRecallMetric,
  ContextualPrecisionMetric,
  AnswerRelevancyMetric,
  FaithfulnessMetric
)



class CustomOllamaLlama(DeepEvalBaseLLM):
    def __init__(self, model_name: str = "llama3.1:8b"):
    # def __init__(self, model_name: str = "mistral"):


        self.model_name = model_name
        self.client = Client()
        self.async_client = AsyncClient()

    def load_model(self):
        return self.model_name

    def generate(self, prompt: str) -> str:
        response = self.client.chat(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0.7, "num_predict": 2500},


        )
        return response["message"]["content"]

    async def a_generate(self, prompt: str) -> str:
        response = await self.async_client.chat(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0.7, "num_predict": 2500},
        )
        return response["message"]["content"]

    def get_model_name(self) -> str:
        return f"Ollama ({self.model_name})"


custom_llm = CustomOllamaLlama(model_name="llama3.1:8b")

correctness_metric = GEval(
    name="Correctness",
    evaluation_steps=[
    "Use the 'context' as the sole ground truth. Ignore outside knowledge and don't assume the output is correct just because it sounds fluent.",
    "Extract each distinct factual claim from the 'actual output'.",
    "For each claim, check it against the 'context': MATCHES, DIFFERS (contradicts/misstates it), or UNVERIFIABLE (context doesn't address it).",
    "Only DIFFERS counts against the score. Do not penalize missing details (that's Recall's job) or UNVERIFIABLE claims.",
    "No DIFFERS claims → high score (close to 1). Any DIFFERS claim → score drops sharply, scaled by how many and how central they are."
    ],
    evaluation_params=[SingleTurnParams.INPUT,SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.CONTEXT],
    threshold=0.5,
    model=custom_llm
)


# Load test cases from the csv
# OSL commented
df = pd.read_csv("/content/drive/MyDrive/Clean Data(LLM Responses ).csv")

test_cases = [
    LLMTestCase(
        input=row["Input"],
        actual_output=row["Actual_output"],
        context=[str(row["Expected_output"])]
    )
    for _, row in df.iterrows()
]
print(f"Loaded {len(test_cases)} test cases")

evaluate(
    test_cases,
    [correctness_metric ]
    # , # run 5 at  a time instead of all at once
       , async_config=AsyncConfig(
        run_async=True,
        max_concurrent=5,      # lower this since each task is slow on your setup
        throttle_value=1       # 1 second delay between task submissions
    )
)

Loaded 72 test cases


✨ You're running DeepEval's latest Correctness [GEval] Metric! (using Ollama (llama3.1:8b), strict=False, 
async_mode=True)...

Output()

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:            Who is this guide for?                                                                 │
│  │     Actual Output:    Since our conversation just started and there's no prior context, my response will     │
│  │                       be brief.                                                                              │
│  │                                                                                                              │
│  │                       However, based on the writing rules you provided earlier (don't over-explain,          │
│  │                       include necessary information, concise, scannable, simple, direct), I can infer        │
│  │                       that this guide is likely intended for a UX writing assistant or a content             │
│  │                       strategist who wants to ensure their microcopy aligns with established guidelines      │
│  │                       and best practices.                                                                    │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric              ┃ Score ┃ Threshold ┃ Reason                                                 │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Correctness [GEval] │ 0.40  │ 0.50      │ The actual output makes a factual claim about the      │
│              │                     │       │           │ guide's intended audience, which DIFFERS from the      │
│              │                     │       │           │ context, as the context explicitly states the guide    │
│              │                     │       │           │ is for product designers, content designers, UX        │
│              │                     │       │           │ writers, and engineers, but the output implies it's    │
│              │                     │       │           │ for a UX writing assistant or content strategist.      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:            What specific UI elements and words that appear in the product does this guide         │
│  │                       cover?                                                                                 │
│  │     Actual Output:    Since we haven't discussed any specific UI elements or product names, my previous      │
│  │                       response was an educated guess based on the writing rules you provided earlier.        │
│  │                                                                                                              │
│  │                       However, I'll try to provide a m

⚠ WARNING: No hyperparameters logged.
» ]8;id=764079;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 417.94s | token cost: None)
» Test Results (106 total tests):
   » Pass Rate: 73.58% | Passed: 78 | Failed: 28

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=False, metrics_data=[MetricData(name='Correctness [GEval]', threshold=0.5, success=False, score=0.4, reason="The actual output makes a factual claim about the guide's intended audience, which DIFFERS from the context, as the context explicitly states the guide is for product designers, content designers, UX writers, and engineers, but the output implies it's for a UX writing assistant or content strategist.", strict_mode=False, flaky=False, evaluation_model='Ollama (llama3.1:8b)', error=None, evaluation_cost=None, input_tokens=None, output_tokens=None, verbose_logs='Criteria:\nNone \n \nEvaluation Steps:\n[\n    "Use the \'context\' as the sole ground truth. Ignore outside knowledge and don\'t assume the output is correct just because it sounds fluent.",\n    "Extract each distinct factual claim from the \'actual output\'.",\n    "For each claim, check it against the \'context\': MATCHES, DIFFERS (contradicts/misstat

# **Correctness - RAG**

In [ ]:

import os
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "50000"
os.environ["DEEPEVAL_TASK_GATHER_BUFFER_SECONDS_OVERRIDE"] = "200"
import asyncio
from ollama import AsyncClient, Client
from deepeval.models import DeepEvalBaseLLM
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.metrics import GEval
# OS New Edits
from deepeval.metrics import (
  ContextualRelevancyMetric,
  ContextualRecallMetric,
  ContextualPrecisionMetric,
  AnswerRelevancyMetric,
  FaithfulnessMetric
)



class CustomOllamaLlama(DeepEvalBaseLLM):
    def __init__(self, model_name: str = "llama3.1:8b"):
    # def __init__(self, model_name: str = "mistral"):


        self.model_name = model_name
        self.client = Client()
        self.async_client = AsyncClient()

    def load_model(self):
        return self.model_name

    def generate(self, prompt: str) -> str:
        response = self.client.chat(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0.7, "num_predict": 2500},


        )
        return response["message"]["content"]

    async def a_generate(self, prompt: str) -> str:
        response = await self.async_client.chat(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0.7, "num_predict": 2500},
        )
        return response["message"]["content"]

    def get_model_name(self) -> str:
        return f"Ollama ({self.model_name})"


custom_llm = CustomOllamaLlama(model_name="llama3.1:8b")
# custom_llm = CustomOllamaLlama(model_name="mistral")

correctness_metric = GEval(
    name="Correctness",
    evaluation_steps=[
    "Use the 'context' as the sole ground truth. Ignore outside knowledge and don't assume the output is correct just because it sounds fluent.",
    "Extract each distinct factual claim from the 'actual output'.",
    "For each claim, check it against the 'context': MATCHES, DIFFERS (contradicts/misstates it), or UNVERIFIABLE (context doesn't address it).",
    "Only DIFFERS counts against the score. Do not penalize missing details (that's Recall's job) or UNVERIFIABLE claims.",
    "No DIFFERS claims → high score (close to 1). Any DIFFERS claim → score drops sharply, scaled by how many and how central they are."
    ],
    evaluation_params=[SingleTurnParams.INPUT,SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.CONTEXT],
    threshold=0.5,
    model=custom_llm
)

# Load test cases from the csv
# OSL commented
df = pd.read_csv("/content/drive/MyDrive/Clean Data(Sheet2).csv")

test_cases = [
    LLMTestCase(
        input=row["Input"],
        actual_output=row["Actual_output"],
        context=[str(row["Expected_output"])]
    )
    for _, row in df.iterrows()
]
print(f"Loaded {len(test_cases)} test cases")

evaluate(
    test_cases,
    [correctness_metric ]
    # , # run 5 at  a time instead of all at once
       , async_config=AsyncConfig(
        run_async=True,
        max_concurrent=5,      # lower this since each task is slow on your setup
        throttle_value=1       # 1 second delay between task submissions
    )
)

Loaded 72 test cases


✨ You're running DeepEval's latest Correctness [GEval] Metric! (using Ollama (llama3.1:8b), strict=False, 
async_mode=True)...

Output()

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:            What specific UI elements and words that appear in the product does this guide         │
│  │                       cover?                                                                                 │
│  │     Actual Output:    The guide covers specific UI elements and words that appear in the product,            │
│  │                       including:                                                                             │
│  │                                                                                                              │
│  │                       * Dialog                                                                               │
│  │                       * Snackbar                                                                             │
│  │                       * Button                                                                               │
│  │                       * Label                                                                                │
│  │                       * Notification                                                                         │
│  │                       * Error messages                                                                       │
│  │                       * Tooltips                                                                             │
│  │                       * Onboarding flows                                                                     │
│  │                       * Various writing styles (e.g. present tense, verb forms)                              │
│  │                       * Common words with no jargon or acronyms                                              │
│  │                                                                                                              │
│  │                       The guide also mentions specific words to avoid, such as:                              │
│  │                                                                                                              │
│  │                       * Visa Secure rules                                                                    │
│  │                       * Accessibility checks including alt text, label, and WCAG AA pass                     │
│  │                                                                                                              │
│  │                       This guide is specifically tailored for UX writers, product designers, content         │
│  │                       designers, engineers, and any team member who writes microcopy in the product.         │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric              ┃ Score ┃ Threshold ┃ Reason                                                 │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━

⚠ WARNING: No hyperparameters logged.
» ]8;id=315364;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 246.91s | token cost: None)
» Test Results (72 total tests):
   » Pass Rate: 95.83% | Passed: 69 | Failed: 3

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Correctness [GEval]', threshold=0.5, success=True, score=0.8, reason="Most factual claims in the actual output match the context, but there are two claims that differ slightly in wording. The output lists 'UX writers (both technical and non-technical)', whereas the context only mentions 'UX writers'. Additionally, the output includes 'content designers' as a separate item, whereas the context groups them with 'product designers'.", strict_mode=False, flaky=False, evaluation_model='Ollama (llama3.1:8b)', error=None, evaluation_cost=None, input_tokens=None, output_tokens=None, verbose_logs='Criteria:\nNone \n \nEvaluation Steps:\n[\n    "Use the \'context\' as the sole ground truth. Ignore outside knowledge and don\'t assume the output is correct just because it sounds fluent.",\n    "Extract each distinct factual claim from the \'actual output\'.",\n    "For each claim, check it ag


# Faithfulness - LLM

In [ ]:

import os
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "50000"
os.environ["DEEPEVAL_TASK_GATHER_BUFFER_SECONDS_OVERRIDE"] = "200"
import asyncio
from ollama import AsyncClient, Client
from deepeval.models import DeepEvalBaseLLM
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.metrics import GEval
# OS New Edits
from deepeval.metrics import (
  ContextualRelevancyMetric,
  ContextualRecallMetric,
  ContextualPrecisionMetric,
  AnswerRelevancyMetric,
  FaithfulnessMetric
)



class CustomOllamaLlama(DeepEvalBaseLLM):
    def __init__(self, model_name: str = "llama3.1:8b"):
    # def __init__(self, model_name: str = "mistral"):


        self.model_name = model_name
        self.client = Client()
        self.async_client = AsyncClient()

    def load_model(self):
        return self.model_name

    def generate(self, prompt: str) -> str:
        response = self.client.chat(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0.7, "num_predict": 2500},


        )
        return response["message"]["content"]

    async def a_generate(self, prompt: str) -> str:
        response = await self.async_client.chat(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0.7, "num_predict": 2500},
        )
        return response["message"]["content"]

    def get_model_name(self) -> str:
        return f"Ollama ({self.model_name})"


custom_llm = CustomOllamaLlama(model_name="llama3.1:8b")

faithfulness_metric = GEval(
    name="Faithfulness",
    evaluation_steps=[
        "Use the 'context' as the sole ground truth, regardless of whether the pipeline retrieved it. Ignore outside knowledge.",
        "For each distinct claim in the 'actual output', check by meaning (not exact wording) whether it is grounded in the 'context'.",
        "Increase the score for claims that are grounded in the context, even if reworded.",
        "Lower the score for claims that are fabricated (not present in or derivable from the context) or that reverse/negate what the context says. Both are equally serious hallucinations.",
        "Do not penalize the output for omitting details from the context — that is evaluated separately by a Contextual Recall metric, not this one.",
        "If every claim is grounded in the context, the score should be high (close to 1). Score should drop sharply as more claims are fabricated or reversed."
    ],
    evaluation_params=[SingleTurnParams.INPUT,SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.CONTEXT],
    threshold=0.5,
    model=custom_llm
)


# Load test cases from the csv
# OSL commented
df = pd.read_csv("/content/drive/MyDrive/Clean Data(LLM Responses ).csv")

test_cases = [
    LLMTestCase(
        input=row["Input"],
        actual_output=row["Actual_output"],
        context=[str(row["Expected_output"])]
    )
    for _, row in df.iterrows()
]
print(f"Loaded {len(test_cases)} test cases")

evaluate(
    test_cases,
    [faithfulness_metric ]
    # , # run 5 at  a time instead of all at once
       , async_config=AsyncConfig(
        run_async=True,
        max_concurrent=5,      # lower this since each task is slow on your setup
        throttle_value=1       # 1 second delay between task submissions
    )
)

Loaded 72 test cases


✨ You're running DeepEval's latest Faithfulness [GEval] Metric! (using Ollama (llama3.1:8b), strict=False, 
async_mode=True)...

Output()

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:            What specific UI elements and words that appear in the product does this guide         │
│  │                       cover?                                                                                 │
│  │     Actual Output:    Since we haven't discussed any specific UI elements or product names, my previous      │
│  │                       response was an educated guess based on the writing rules you provided earlier.        │
│  │                                                                                                              │
│  │                       However, I'll try to provide a more concrete answer now. This guide likely covers      │
│  │                       microcopy related to navigation, input validation, error messages, and other common    │
│  │                       UI elements that are frequently used in digital products. If you'd like to share       │
│  │                       some specific context or product names, I can try to tailor my response                │
│  │                       accordingly!                                                                           │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Faithfulness [GEval] │ 0.20  │ 0.50      │ The actual output makes claims about UI elements      │
│              │                      │       │           │ and words covered by the guide, but these claims      │
│              │                      │       │           │ are not grounded in the context. The output           │
│              │                      │       │           │ attempts to provide educated guesses and reworded     │
│              │                      │       │           │ claims, but these are not supported by the provided   │
│              │                      │       │           │ context.                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_3 (Passed 1 metrics)                          

⚠ WARNING: No hyperparameters logged.
» ]8;id=774786;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 414.4s | token cost: None)
» Test Results (116 total tests):
   » Pass Rate: 80.17% | Passed: 93 | Failed: 23

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Faithfulness [GEval]', threshold=0.5, success=True, score=0.8, reason='Most claims in the Actual Output are grounded in the context, even if reworded, and no claims are fabricated or reversed. However, the score could be higher if all claims exactly matched the context, and the output omits some details from the context, which would be evaluated separately.', strict_mode=False, flaky=False, evaluation_model='Ollama (llama3.1:8b)', error=None, evaluation_cost=None, input_tokens=None, output_tokens=None, verbose_logs='Criteria:\nNone \n \nEvaluation Steps:\n[\n    "Use the \'context\' as the sole ground truth, regardless of whether the pipeline retrieved it. Ignore outside knowledge.",\n    "For each distinct claim in the \'actual output\', check by meaning (not exact wording) whether it is grounded in the \'context\'.",\n    "Increase the score for claims that are grounded in the c

# Faithfulness - RAG

In [ ]:

import os
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "50000"
os.environ["DEEPEVAL_TASK_GATHER_BUFFER_SECONDS_OVERRIDE"] = "200"
import asyncio
from ollama import AsyncClient, Client
from deepeval.models import DeepEvalBaseLLM
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.metrics import GEval
# OS New Edits
from deepeval.metrics import (
  ContextualRelevancyMetric,
  ContextualRecallMetric,
  ContextualPrecisionMetric,
  AnswerRelevancyMetric,
  FaithfulnessMetric
)



class CustomOllamaLlama(DeepEvalBaseLLM):
    def __init__(self, model_name: str = "llama3.1:8b"):
    # def __init__(self, model_name: str = "mistral"):


        self.model_name = model_name
        self.client = Client()
        self.async_client = AsyncClient()

    def load_model(self):
        return self.model_name

    def generate(self, prompt: str) -> str:
        response = self.client.chat(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0.7, "num_predict": 2500},


        )
        return response["message"]["content"]

    async def a_generate(self, prompt: str) -> str:
        response = await self.async_client.chat(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0.7, "num_predict": 2500},
        )
        return response["message"]["content"]

    def get_model_name(self) -> str:
        return f"Ollama ({self.model_name})"


custom_llm = CustomOllamaLlama(model_name="llama3.1:8b")

faithfulness_metric = GEval(
    name="Faithfulness",
    evaluation_steps=[
        "Use the 'context' as the sole ground truth, regardless of whether the pipeline retrieved it. Ignore outside knowledge.",
        "For each distinct claim in the 'actual output', check by meaning (not exact wording) whether it is grounded in the 'context'.",
        "Increase the score for claims that are grounded in the context, even if reworded.",
        "Lower the score for claims that are fabricated (not present in or derivable from the context) or that reverse/negate what the context says. Both are equally serious hallucinations.",
        "Do not penalize the output for omitting details from the context — that is evaluated separately by a Contextual Recall metric, not this one.",
        "If every claim is grounded in the context, the score should be high (close to 1). Score should drop sharply as more claims are fabricated or reversed."
    ],
    evaluation_params=[SingleTurnParams.INPUT,SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.CONTEXT],
    threshold=0.5,
    model=custom_llm
)

# Load test cases from the csv
# OSL commented
df = pd.read_csv("/content/drive/MyDrive/Clean Data(Sheet2).csv")

test_cases = [
    LLMTestCase(
        input=row["Input"],
        actual_output=row["Actual_output"],
        context=[str(row["Expected_output"])]
    )
    for _, row in df.iterrows()
]
print(f"Loaded {len(test_cases)} test cases")

evaluate(
    test_cases,
    [faithfulness_metric ]
    # , # run 5 at  a time instead of all at once
       , async_config=AsyncConfig(
        run_async=True,
        max_concurrent=5,      # lower this since each task is slow on your setup
        throttle_value=1       # 1 second delay between task submissions
    )
)

Loaded 72 test cases


✨ You're running DeepEval's latest Faithfulness [GEval] Metric! (using Ollama (llama3.1:8b), strict=False, 
async_mode=True)...

Output()

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_3 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_4 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_5 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_6 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_7 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_8 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_9 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_10 (Passed 1 metrics)                                

⚠ WARNING: No hyperparameters logged.
» ]8;id=153333;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 255.2s | token cost: None)
» Test Results (72 total tests):
   » Pass Rate: 97.22% | Passed: 70 | Failed: 2

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Faithfulness [GEval]', threshold=0.5, success=True, score=0.8, reason='Most claims in the actual output are well-grounded in the context, with some rewording and minor omissions.', strict_mode=False, flaky=False, evaluation_model='Ollama (llama3.1:8b)', error=None, evaluation_cost=None, input_tokens=None, output_tokens=None, verbose_logs='Criteria:\nNone \n \nEvaluation Steps:\n[\n    "Use the \'context\' as the sole ground truth, regardless of whether the pipeline retrieved it. Ignore outside knowledge.",\n    "For each distinct claim in the \'actual output\', check by meaning (not exact wording) whether it is grounded in the \'context\'.",\n    "Increase the score for claims that are grounded in the context, even if reworded.",\n    "Lower the score for claims that are fabricated (not present in or derivable from the context) or that reverse/negate what the context says. Both ar

# Answer Relevancy - LLM

In [ ]:

import os
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "50000"
os.environ["DEEPEVAL_TASK_GATHER_BUFFER_SECONDS_OVERRIDE"] = "200"
import asyncio
from ollama import AsyncClient, Client
from deepeval.models import DeepEvalBaseLLM
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.metrics import GEval
# OS New Edits
from deepeval.metrics import (
  ContextualRelevancyMetric,
  ContextualRecallMetric,
  ContextualPrecisionMetric,
  AnswerRelevancyMetric,
  FaithfulnessMetric
)



class CustomOllamaLlama(DeepEvalBaseLLM):
    def __init__(self, model_name: str = "llama3.1:8b"):
    # def __init__(self, model_name: str = "mistral"):


        self.model_name = model_name
        self.client = Client()
        self.async_client = AsyncClient()

    def load_model(self):
        return self.model_name

    def generate(self, prompt: str) -> str:
        response = self.client.chat(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0.7, "num_predict": 2500},


        )
        return response["message"]["content"]

    async def a_generate(self, prompt: str) -> str:
        response = await self.async_client.chat(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0.7, "num_predict": 2500},
        )
        return response["message"]["content"]

    def get_model_name(self) -> str:
        return f"Ollama ({self.model_name})"


custom_llm = CustomOllamaLlama(model_name="llama3.1:8b")


answer_relevancy_metric = GEval(
    name="Answer Relevancy",
    evaluation_steps=[
        "This metric evaluates topical relevance only — whether the 'actual output' addresses what the 'input' is actually asking.",
        "Identify what the 'input' task prompt is specifically asking for.",
        "Assess what proportion of the 'actual output' directly engages with that specific ask, versus content that is off-topic, tangential, broader than asked, or padded.",
        "Do not factor in factual accuracy — that is evaluated separately by the Correctness metric, not this one.",
        "Do not factor in completeness of coverage — that is evaluated separately by the Contextual Recall metric, not this one.",
        "Base the score on the proportion of the actual output that is genuinely on-topic and responsive, not solely on whether any off-topic content is severe enough to notice."
    ],
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
    threshold=0.5,
    model=custom_llm
)

# Load test cases from the csv
# OSL commented
df = pd.read_csv("/content/drive/MyDrive/Clean Data(LLM Responses ).csv")

test_cases = [
    LLMTestCase(
        input=row["Input"],
        actual_output=row["Actual_output"],
        context=[str(row["Expected_output"])]
    )
    for _, row in df.iterrows()
]
print(f"Loaded {len(test_cases)} test cases")

evaluate(
    test_cases,
    [answer_relevancy_metric ]
    # , # run 5 at  a time instead of all at once
       , async_config=AsyncConfig(
        run_async=True,
        max_concurrent=5,      # lower this since each task is slow on your setup
        throttle_value=1       # 1 second delay between task submissions
    )
)

Loaded 72 test cases


✨ You're running DeepEval's latest Answer Relevancy [GEval] Metric! (using Ollama (llama3.1:8b), strict=False, 
async_mode=True)...

Output()

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:            What specific UI elements and words that appear in the product does this guide         │
│  │                       cover?                                                                                 │
│  │     Actual Output:    Since we haven't discussed any specific UI elements or product names, my previous      │
│  │                       response was an educated guess based on the writing rules you provided earlier.        │
│  │                                                                                                              │
│  │                       However, I'll try to provide a more concrete answer now. This guide likely covers      │
│  │                       microcopy related to navigation, input validation, error messages, and other common    │
│  │                       UI elements that are frequently used in digital products. If you'd like to share       │
│  │                       some specific context or product names, I can try to tailor my response                │
│  │                       accordingly!                                                                           │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric                   ┃ Score ┃ Threshold ┃ Reason                                            │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Answer Relevancy [GEval] │ 0.40  │ 0.50      │ The actual output partially addresses the         │
│              │                          │       │           │ input's ask, but it starts with a defensive       │
│              │                          │       │           │ statement and then provides a vague and general   │
│              │                          │       │           │ answer, without directly answering the question   │
│              │                          │       │           │ about specific UI elements and words covered in   │
│              │                          │       │           │ the guide.                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_3 (Passed 1 metrics)                          

⚠ WARNING: No hyperparameters logged.
» ]8;id=908185;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 213.37s | token cost: None)
» Test Results (72 total tests):
   » Pass Rate: 93.06% | Passed: 67 | Failed: 5

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy [GEval]', threshold=0.5, success=True, score=0.8, reason='The response directly engages with the input task prompt by inferring the intended audience for the guide, but the engagement is not entirely on-topic as it relies on prior context and writing rules not provided in the input.', strict_mode=False, flaky=False, evaluation_model='Ollama (llama3.1:8b)', error=None, evaluation_cost=None, input_tokens=None, output_tokens=None, verbose_logs='Criteria:\nNone \n \nEvaluation Steps:\n[\n    "This metric evaluates topical relevance only — whether the \'actual output\' addresses what the \'input\' is actually asking.",\n    "Identify what the \'input\' task prompt is specifically asking for.",\n    "Assess what proportion of the \'actual output\' directly engages with that specific ask, versus content that is off-topic, tangential, broader than asked, or padded.",\n   

# Answer Relevancy - RAG

In [ ]:

import os
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "50000"
os.environ["DEEPEVAL_TASK_GATHER_BUFFER_SECONDS_OVERRIDE"] = "200"
import asyncio
from ollama import AsyncClient, Client
from deepeval.models import DeepEvalBaseLLM
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.metrics import GEval
# OS New Edits
from deepeval.metrics import (
  ContextualRelevancyMetric,
  ContextualRecallMetric,
  ContextualPrecisionMetric,
  AnswerRelevancyMetric,
  FaithfulnessMetric
)



class CustomOllamaLlama(DeepEvalBaseLLM):
    def __init__(self, model_name: str = "llama3.1:8b"):
    # def __init__(self, model_name: str = "mistral"):


        self.model_name = model_name
        self.client = Client()
        self.async_client = AsyncClient()

    def load_model(self):
        return self.model_name

    def generate(self, prompt: str) -> str:
        response = self.client.chat(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0.7, "num_predict": 2500},


        )
        return response["message"]["content"]

    async def a_generate(self, prompt: str) -> str:
        response = await self.async_client.chat(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0.7, "num_predict": 2500},
        )
        return response["message"]["content"]

    def get_model_name(self) -> str:
        return f"Ollama ({self.model_name})"


custom_llm = CustomOllamaLlama(model_name="llama3.1:8b")


answer_relevancy_metric = GEval(
    name="Answer Relevancy",
    evaluation_steps=[
        "This metric evaluates topical relevance only — whether the 'actual output' addresses what the 'input' is actually asking.",
        "Identify what the 'input' task prompt is specifically asking for.",
        "Assess what proportion of the 'actual output' directly engages with that specific ask, versus content that is off-topic, tangential, broader than asked, or padded.",
        "Do not factor in factual accuracy — that is evaluated separately by the Correctness metric, not this one.",
        "Do not factor in completeness of coverage — that is evaluated separately by the Contextual Recall metric, not this one.",
        "Base the score on the proportion of the actual output that is genuinely on-topic and responsive, not solely on whether any off-topic content is severe enough to notice."
    ],
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
    threshold=0.5,
    model=custom_llm
)


# Load test cases from the csv
# OSL commented

df = pd.read_csv("/content/drive/MyDrive/Clean Data(Sheet2).csv")

test_cases = [
    LLMTestCase(
        input=row["Input"],
        actual_output=row["Actual_output"],
        context=[str(row["Expected_output"])]
    )
    for _, row in df.iterrows()
]
print(f"Loaded {len(test_cases)} test cases")

evaluate(
    test_cases,
    [answer_relevancy_metric ]
    # , # run 5 at  a time instead of all at once
       , async_config=AsyncConfig(
        run_async=True,
        max_concurrent=5,      # lower this since each task is slow on your setup
        throttle_value=1       # 1 second delay between task submissions
    )
)

Loaded 72 test cases


✨ You're running DeepEval's latest Answer Relevancy [GEval] Metric! (using Ollama (llama3.1:8b), strict=False, 
async_mode=True)...

Output()

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_3 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_4 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_5 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_6 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_7 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_8 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_9 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_10 (Passed 1 metrics)                                

⚠ WARNING: No hyperparameters logged.
» ]8;id=694891;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 197.67s | token cost: None)
» Test Results (72 total tests):
   » Pass Rate: 100.0% | Passed: 72 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy [GEval]', threshold=0.5, success=True, score=1.0, reason="The actual output directly addresses the input task prompt and provides a clear and specific list of roles and team members who can benefit from this guide, engaging with the prompt's ask.", strict_mode=False, flaky=False, evaluation_model='Ollama (llama3.1:8b)', error=None, evaluation_cost=None, input_tokens=None, output_tokens=None, verbose_logs='Criteria:\nNone \n \nEvaluation Steps:\n[\n    "This metric evaluates topical relevance only — whether the \'actual output\' addresses what the \'input\' is actually asking.",\n    "Identify what the \'input\' task prompt is specifically asking for.",\n    "Assess what proportion of the \'actual output\' directly engages with that specific ask, versus content that is off-topic, tangential, broader than asked, or padded.",\n    "Do not factor in factual accuracy —